# バトミントンのラケット選び
バトミントンでのスマッシュの姿勢を分析解析してアドバイスをするアプリの試作品

In [ ]:
!nvidia-smi

In [ ]:
# Mounting the drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
work_dir = '/content/drive/MyDrive/プレゼン発表/岐阜BLUVIC_20260516'
%ls /content/drive/MyDrive/プレゼン発表/岐阜BLUVIC_20260516

In [ ]:
!pip install ultralytics opencv-python pandas numpy matplotlib -q

In [ ]:
from ultralytics import YOLO
import cv2
import pandas as pd
import numpy as np
import os
from pathlib import Path

# =========================
# 1. 入力動画の指定
# =========================

work_dir = '/content/drive/MyDrive/プレゼン発表/岐阜BLUVIC_20260516'
input_video = f'{work_dir}/クリア.MOV'

print("input_video:", input_video)
print("exists:", os.path.exists(input_video))

# =========================
# 2. 出力フォルダ
# =========================

output_dir = f'{work_dir}/outputs'
os.makedirs(output_dir, exist_ok=True)

# =========================
# 3. MOVをMP4に変換
#    OpenCVでMOVが読めない場合があるため
# =========================

converted_video = f'{output_dir}/clear_input.mp4'

if not os.path.exists(converted_video):
    !ffmpeg -y -i "$input_video" -vcodec libx264 -pix_fmt yuv420p -acodec aac "$converted_video"

video_path = converted_video

print("video_path:", video_path)
print("converted exists:", os.path.exists(video_path))

# =========================
# 4. YOLOv8 Pose モデル
# =========================

model = YOLO("yolov8n-pose.pt")

# =========================
# 5. 動画読み込み
# =========================

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError(f"動画を開けませんでした: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# FPSが取れない場合の保険
if fps is None or fps <= 0:
    fps = 30.0

print("fps:", fps)
print("width:", width)
print("height:", height)
print("total_frames:", total_frames)

# =========================
# 6. 出力動画設定
# =========================

output_video_path = f"{output_dir}/overlay_pose.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

if not out.isOpened():
    raise RuntimeError("出力動画ファイルを作成できませんでした。")

rows = []
frame_idx = 0

# Colab GPUなら1でOK。CPUで遅ければ2か3にする
frame_skip = 1

# =========================
# 7. YOLOv8-Pose 推論ループ
# =========================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # スキップするフレームは元動画のまま保存
    if frame_idx % frame_skip != 0:
        out.write(frame)
        frame_idx += 1
        continue

    results = model(frame, verbose=False)
    r = results[0]

    # 骨格つきフレーム
    annotated_frame = r.plot()

    # サイズがずれる場合の保険
    if annotated_frame.shape[1] != width or annotated_frame.shape[0] != height:
        annotated_frame = cv2.resize(annotated_frame, (width, height))

    out.write(annotated_frame)

    # キーポイント保存
    if r.keypoints is not None and r.keypoints.xy is not None:
        keypoints = r.keypoints.xy.cpu().numpy()
        confs = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else None
        boxes = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else None

        if len(keypoints) > 0:
            # 複数人が映った場合、一番大きく映っている人物を選ぶ
            if boxes is not None and len(boxes) > 0:
                areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
                person_idx = int(np.argmax(areas))
            else:
                person_idx = 0

            person = keypoints[person_idx]
            person_conf = confs[person_idx] if confs is not None else np.ones(17)

            row = {
                "frame": frame_idx,
                "time_sec": frame_idx / fps
            }

            for i, (x, y) in enumerate(person):
                row[f"kpt_{i}_x"] = float(x)
                row[f"kpt_{i}_y"] = float(y)
                row[f"kpt_{i}_conf"] = float(person_conf[i])

            rows.append(row)

    if frame_idx % 50 == 0:
        print(f"processed {frame_idx}/{total_frames}")

    frame_idx += 1

cap.release()
out.release()

# =========================
# 8. CSV保存
# =========================

df = pd.DataFrame(rows)
csv_path = f"{output_dir}/keypoints.csv"
df.to_csv(csv_path, index=False)

print("完了")
print("骨格つき動画:", output_video_path)
print("キーポイントCSV:", csv_path)
print("検出できたフレーム数:", len(df))

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

video_file = output_video_path

print("video_file:", video_file)
print("exists:", os.path.exists(video_file))

mp4 = open(video_file, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width="360" controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")

In [ ]:
# =========================
# OpenAIに送る準備
# 1. 骨格つき動画から5枚切り出し
# 2. keypoints.csvを読み込み
# 3. 画像5枚 + CSV要約をOpenAIへ送信
# =========================

!pip -q install openai

import os
import cv2
import base64
import pandas as pd
from IPython.display import display, Image, Markdown
from getpass import getpass
from openai import OpenAI

# =========================
# 1. OpenAI API Key
# =========================

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()


client = OpenAI()
# =========================
# 2. 骨格つき動画から5枚切り出し
# =========================

video_file = output_video_path   # すでに作成済み: overlay_pose.mp4
frame_dir = f"{output_dir}/openai_frames"
os.makedirs(frame_dir, exist_ok=True)

cap = cv2.VideoCapture(video_file)

if not cap.isOpened():
    raise RuntimeError(f"動画を開けませんでした: {video_file}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("video_file:", video_file)
print("total_frames:", total_frames)
print("fps:", fps)

# クリア動作全体をざっくり見るために5枚
target_frames = [
    int(total_frames * 0.20),
    int(total_frames * 0.35),
    int(total_frames * 0.50),
    int(total_frames * 0.65),
    int(total_frames * 0.80),
]

saved_paths = []

for f in target_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ret, frame = cap.read()

    if ret:
        path = f"{frame_dir}/frame_{f}.jpg"

        # OpenAIに送るので少し軽くする
        # 縦長動画なので幅を720px程度に縮小
        h, w = frame.shape[:2]
        new_w = 720
        new_h = int(h * new_w / w)
        resized = cv2.resize(frame, (new_w, new_h))

        cv2.imwrite(path, resized, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
        saved_paths.append(path)
        print("saved:", path)

cap.release()

# 確認表示
for path in saved_paths:
    display(Image(filename=path, width=250))

In [ ]:
# =========================
# 3. CSV読み込み
# =========================

df = pd.read_csv(csv_path)

print("CSV shape:", df.shape)
display(df.head())

# CSV全体をそのまま投げると大きすぎるので、
# まずは情報を要約して送る
csv_summary = {
    "rows": len(df),
    "columns": list(df.columns),
    "first_frame": int(df["frame"].min()),
    "last_frame": int(df["frame"].max()),
    "start_time_sec": float(df["time_sec"].min()),
    "end_time_sec": float(df["time_sec"].max()),
}

# 代表行だけCSV文字列にする
sample_df = df.iloc[
    [
        int(len(df) * 0.20),
        int(len(df) * 0.35),
        int(len(df) * 0.50),
        int(len(df) * 0.65),
        int(len(df) * 0.80),
    ]
]

sample_csv_text = sample_df.to_csv(index=False)

print("CSV summary:")
print(csv_summary)
print()
print("sample_csv_text:")
print(sample_csv_text[:2000])

In [ ]:
# =========================
# 4. 画像をbase64に変換
# =========================

def image_to_data_url(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


content = []

prompt_text = f"""
あなたはバドミントンのフォーム解析コーチです。

以下は、バドミントンの「クリア」動作をYOLOv8 Poseで姿勢推定した結果です。

入力:
1. 骨格つき動画から切り出した代表フレーム5枚
2. keypoints.csv の要約
3. 代表フレーム付近のキーポイントCSV

見てほしいこと:
- クリア動作としてフォームがどう見えるか
- 肩、肘、手首の使い方
- 体幹の傾き
- 膝、足、重心の使い方
- インパクト位置の推定
- 初心者にもわかる改善点
- 次に数値解析すべき項目

注意:
- 画像は骨格推定済みのフレームです。
- CSVのキーポイント番号はYOLOv8 Poseの17点形式です。
- 断定しすぎず、画像から見える範囲で評価してください。

CSV summary:
{csv_summary}

Representative keypoints CSV:
{sample_csv_text}
"""

content.append({
    "type": "input_text",
    "text": prompt_text
})

for path in saved_paths:
    content.append({
        "type": "input_image",
        "image_url": image_to_data_url(path)
    })

# =========================
# 5. OpenAI APIへ送信
# =========================

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": content
        }
    ],
)

analysis_text = response.output_text

display(Markdown(analysis_text))

In [ ]:
import json
import pandas as pd
from IPython.display import display

product_json_path = f"{work_dir}/yonex-database.json"

with open(product_json_path, "r", encoding="utf-8") as f:
    yonex_db = json.load(f)

rackets = yonex_db.get("rackets", [])
strings = yonex_db.get("strings", [])

print("ラケット数:", len(rackets))
print("ガット情報数:", len(strings))

rackets_df = pd.DataFrame(rackets)
strings_df = pd.DataFrame(strings)

display(rackets_df.head())
display(strings_df.head())

In [ ]:
products_json_text = json.dumps(
    {
        "rackets": rackets,
        "strings": strings
    },
    ensure_ascii=False,
    indent=2
)

print(products_json_text[:3000])

In [ ]:
import base64
import json
from IPython.display import Markdown, display

# =========================
# 1. 画像をbase64に変換
# =========================

def image_to_data_url(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


# =========================
# 2. OpenAIへ送る content を作成
# =========================

content = []

prompt_text = f"""
あなたはバドミントンのフォーム解析コーチ兼ラケット・ガット選定アドバイザーです。

以下の情報をもとに、バドミントンの「クリア」動作のフォーム解析を行い、
さらにYONEX商品データベースの中から、この選手に合うラケットとガットを推薦してください。

# 入力情報

1. 骨格つき動画から切り出した代表フレーム画像
2. keypoints.csv の要約
3. 代表フレーム付近のキーポイントCSV
4. YONEXバドミントン商品データベースJSON
   - rackets: ラケット候補
   - strings: ガット候補

# フォーム解析で見てほしいこと

- クリア動作としてフォームがどう見えるか
- 肩、肘、手首の使い方
- 体幹の傾き
- 膝、足、重心の使い方
- インパクト位置の推定
- 初心者にもわかる改善点
- 次に数値解析すべき項目

# ラケット推薦で重視する観点

フォーム解析結果をもとに、商品データベースの中から合いそうなラケットを選んでください。

特に重視すること:
- クリアが奥まで飛びやすい
- 初心者〜中級者でも扱いやすい
- 重すぎない
- ヘッドが重すぎて振り遅れない
- シャフトや打感が硬すぎない
- コントロールしやすい
- フォーム改善中の選手に合う

# ガット推薦で重視する観点

- クリアを楽に飛ばせるか
- 肩や肘への負担が少ないか
- 初心者〜中級者でも扱いやすいか
- テンションを高くしすぎないこと
- 必要に応じて、推奨テンションも提案すること

# 出力形式

以下の順番で日本語で出力してください。

## 1. フォーム全体の評価
画像とCSVから見える範囲で、断定しすぎずに評価してください。

## 2. 良い点
箇条書きで書いてください。

## 3. 改善点
初心者にもわかるように書いてください。

## 4. ラケット選びの方針
このフォームの選手には、どんな特徴のラケットが合うか説明してください。

## 5. YONEX DBからのおすすめラケット上位3本
それぞれについて、以下を書いてください。

- 商品名
- 型番
- おすすめ度
- おすすめ理由
- フォームとの関係
- 注意点
- 合うプレースタイル

## 6. 一番おすすめのラケット1本
最後に1本だけ選んでください。

## 7. おすすめガット上位2〜3本
それぞれについて、以下を書いてください。

- ガット名またはタイプ
- おすすめ理由
- 推奨テンション
- 注意点

## 8. 最終おすすめセット
例:
ラケット: ○○
ガット: ○○
テンション: ○○ポンド

## 9. 次に追加するとよい解析
例: 肘角度、肩角度、膝角度、インパクト時刻、重心移動など。

# 注意

- 画像は骨格推定済みのフレームです。
- CSVのキーポイント番号はYOLOv8 Poseの17点形式です。
- 商品情報にないスペックは推測しすぎないでください。
- ただし、positioning_note や recommended_tension から自然に読み取れる範囲の判断はしてよいです。
- 必ずYONEX商品データベースJSONの中から推薦してください。
- フォーム解析は画像とCSVから見える範囲に限定し、断定しすぎないでください。

# keypoints.csv summary

{csv_summary}

# Representative keypoints CSV

{sample_csv_text}

# YONEX商品データベースJSON

{products_json_text}
"""

content.append({
    "type": "input_text",
    "text": prompt_text
})

for path in saved_paths:
    content.append({
        "type": "input_image",
        "image_url": image_to_data_url(path)
    })


# =========================
# 3. OpenAI APIへ送信
# =========================

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": content
        }
    ],
)

analysis_text = response.output_text

display(Markdown(analysis_text))